<a href="https://colab.research.google.com/github/Goldeno10/flexisaf_Internship_GenAI_DS_Intermediate/blob/main/task_4/Image_Captioning_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# Image Captioning with CNN Encoder + LSTM Decoder (Flickr8k)
# =============================================================================

# ── 0. Dependencies ──────────────────────────────────────────────────────────
# Run once in your environment before executing this script:
#   pip install opendatasets torchmetrics torch torchvision pandas scikit-learn tqdm matplotlib pillow

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torchvision.transforms as transforms
from torchvision import models
from sklearn.model_selection import train_test_split

try:
    from torchmetrics.functional.text import bleu_score
except:
    ! pip install torchmetrics --quiet
    from torchmetrics.functional.text import bleu_score


# ── 1. Device ─────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 46.9 MB/s eta 0:00:00
Using device: cuda


In [2]:
# ── 2. Download Dataset ───────────────────────────────────────────────────────
try:
    import opendatasets as od
    print("Successfully imported opendatasets")
except:
    ! pip install opendatasets --quiet
    import opendatasets as od
    print("Successfully installed opendatasets")

od.download("https://www.kaggle.com/datasets/adityajn105/flickr8k")


Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: elgeniusgoldeno
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k


100%|██████████| 1.04G/1.04G [00:57<00:00, 19.4MB/s]


In [3]:
# ── 3. Image Transforms ───────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [4]:
# ── 4. Vocabulary ─────────────────────────────────────────────────────────────
class Vocabulary:
    """
    Word ↔ index mapping.

    Save as a .pkl file so you can reuse it across runs or swap it when
    changing datasets.

    Args:
        freq_threshold (int): Minimum word frequency to be included.
    """

    SPECIAL = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}

    def __init__(self, freq_threshold: int = 5):
        self.itos = dict(self.SPECIAL)
        self.stoi = {v: k for k, v in self.itos.items()}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.stoi)

    def build_vocabulary(self, sentence_list: list[str]):
        """Build vocab from a list of raw caption strings."""
        frequencies: Counter = Counter()
        idx = len(self.SPECIAL)  # next available index after special tokens
        for sentence in sentence_list:
            for word in sentence.lower().split():
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text: str) -> list[int]:
        """Convert a raw string to a list of token indices."""
        unk_idx = self.stoi["<UNK>"]
        return [self.stoi.get(token, unk_idx) for token in text.lower().split()]

    def save(self, path: str):
        with open(path, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, path: str) -> "Vocabulary":
        with open(path, "rb") as f:
            return pickle.load(f)


# ── 5. Dataset ────────────────────────────────────────────────────────────────
class ImageCaptionDataset(Dataset):
    """
    Flexible image-caption dataset (Flickr8k / Flickr30k / MS-COCO).

    Args:
        root_dir:    Directory containing the image files.
        captions_df: DataFrame with columns ``image`` and ``caption``.
        transform:   Optional torchvision transform pipeline.
        vocab:       Pre-built Vocabulary; one is built from ``captions_df``
                     if not provided.
    """

    def __init__(self, root_dir, captions_df, transform=None, vocab=None):
        self.root_dir = root_dir
        self.df = captions_df.reset_index(drop=True)
        self.transform = transform

        if vocab is not None:
            self.vocab = vocab
        else:
            self.vocab = Vocabulary(freq_threshold=5)
            self.vocab.build_vocabulary(self.df["caption"].tolist())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        img = Image.open(os.path.join(self.root_dir, row["image"])).convert("RGB")
        if self.transform:
            img = self.transform(img)

        tokens = (
            [self.vocab.stoi["<SOS>"]]
            + self.vocab.numericalize(row["caption"])
            + [self.vocab.stoi["<EOS>"]]
        )
        return img, torch.tensor(tokens, dtype=torch.long)


# ── 6. Collate ────────────────────────────────────────────────────────────────
class MyCollate:
    """Dynamically pad caption sequences inside a batch."""

    def __init__(self, pad_idx: int):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs    = torch.stack([item[0] for item in batch])
        targets = pad_sequence(
            [item[1] for item in batch],
            batch_first=True,
            padding_value=self.pad_idx,
        )
        return imgs, targets


# ── 7. Visualisation helper ───────────────────────────────────────────────────
def show_examples(dataset, num: int = 9):
    """Plot a grid of image + caption examples from *dataset*."""
    mean = np.array(IMAGENET_MEAN)
    std  = np.array(IMAGENET_STD)

    plt.figure(figsize=(18, 15))
    for i in range(num):
        img_t, caption = dataset[i]
        img = std * img_t.permute(1, 2, 0).numpy() + mean
        img = np.clip(img, 0, 1)

        words = [dataset.vocab.itos[t.item()] for t in caption]
        text  = " ".join(w for w in words if w not in {"<PAD>", "<SOS>", "<EOS>"})

        plt.subplot(3, 3, i + 1)
        plt.imshow(img)
        plt.title(text, fontsize=8)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [5]:
# ── 8. Model ──────────────────────────────────────────────────────────────────
class EncoderCNN(nn.Module):
    """
    ResNet-50 encoder with a trainable projection head.

    The backbone weights are frozen by default; only the final linear layer
    (projecting to ``embed_size``) is trained unless ``train_CNN=True``.

    Args:
        embed_size: Output feature dimensionality.
        train_CNN:  Unfreeze backbone weights when ``True``.
    """

    def __init__(self, embed_size: int, train_CNN: bool = False):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for param in resnet.parameters():
            param.requires_grad = train_CNN
        resnet.fc = nn.Linear(resnet.fc.in_features, embed_size)
        resnet.fc.weight.requires_grad_(True)
        resnet.fc.bias.requires_grad_(True)

        self.resnet   = resnet
        self.relu     = nn.ReLU()
        self.dropout  = nn.Dropout(0.5)

    def forward(self, images):
        """Returns image features of shape ``[batch, embed_size]``."""
        return self.dropout(self.relu(self.resnet(images)))


class DecoderRNN(nn.Module):
    """
    LSTM decoder that generates word sequences from image features.

    Args:
        embed_size:  Word embedding dimensionality.
        hidden_size: LSTM hidden state dimensionality.
        vocab_size:  Total vocabulary size (output logits).
        num_layers:  Number of stacked LSTM layers.
    """

    def __init__(self, embed_size: int, hidden_size: int, vocab_size: int, num_layers: int = 1):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, embed_size)
        self.lstm    = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear  = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):
        """
        Teacher-forcing forward pass.

        Args:
            features: Image features ``[batch, embed_size]``.
            captions: Token indices ``[batch, seq_len]`` (batch-first).

        Returns:
            Logits ``[batch, 1 + seq_len, vocab_size]``.
        """
        embeddings = self.dropout(self.embed(captions))                      # [B, T, E]
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)   # [B, 1+T, E]
        hiddens, _ = self.lstm(embeddings)                                    # [B, 1+T, H]
        return self.linear(hiddens)                                           # [B, 1+T, V]


class CNNtoRNN(nn.Module):
    """End-to-end image captioning model."""

    def __init__(self, embed_size: int, hidden_size: int, vocab_size: int, num_layers: int = 1):
        super().__init__()
        self.encoderCNN = EncoderCNN(embed_size)
        self.decoderRNN = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

    def forward(self, images, captions):
        features = self.encoderCNN(images)
        return self.decoderRNN(features, captions)

In [6]:
# ── 9. Beam Search ────────────────────────────────────────────────────────────
@torch.no_grad()
def generate_caption_beam_search(
    image,           # PIL.Image *or* pre-transformed tensor [C, H, W]
    model: nn.Module,
    transform,
    vocab: Vocabulary,
    device,
    beam_size: int = 5,
    max_length: int = 50,
) -> list[str]:
    """
    Generate a caption using beam search.

    Args:
        image:      A PIL ``Image`` (will be transformed) or an already-
                    transformed ``[C, H, W]`` tensor.
        model:      Trained ``CNNtoRNN``.
        transform:  The same transform used during training.
        vocab:      Vocabulary for index ↔ word mapping.
        device:     Torch device.
        beam_size:  Number of active beams.
        max_length: Hard cap on generated sequence length.

    Returns:
        List of word strings (special tokens excluded).
    """
    model.eval()

    # Accept both PIL images and already-transformed tensors
    if isinstance(image, Image.Image):
        img_tensor = transform(image).unsqueeze(0).to(device)   # [1, C, H, W]
    else:
        img_tensor = image.unsqueeze(0).to(device) if image.dim() == 3 else image.to(device)

    features = model.encoderCNN(img_tensor).unsqueeze(1)        # [1, 1, E]
    _, states = model.decoderRNN.lstm(features)

    sos_idx = vocab.stoi["<SOS>"]
    eos_idx = vocab.stoi["<EOS>"]

    # Each beam: (cumulative_log_prob, token_list, lstm_states)
    beams: list[tuple] = [(0.0, [sos_idx], states)]
    completed: list[tuple] = []

    for _ in range(max_length):
        new_beams = []
        for score, caption, last_states in beams:
            last_word = torch.tensor([[caption[-1]]], device=device)    # [1, 1]
            embeds = model.decoderRNN.embed(last_word)                  # [1, 1, E]
            output, curr_states = model.decoderRNN.lstm(embeds, last_states)
            logits = model.decoderRNN.linear(output.squeeze(1))         # [1, V]
            log_probs = F.log_softmax(logits, dim=-1)

            top_log_probs, top_idx = log_probs.topk(beam_size, dim=-1)  # [1, K]

            for k in range(beam_size):
                next_word  = top_idx[0, k].item()
                new_score  = score + top_log_probs[0, k].item()
                new_caption = caption + [next_word]

                if next_word == eos_idx:
                    completed.append((new_score, new_caption))
                else:
                    new_beams.append((new_score, new_caption, curr_states))

        # Prune to top beam_size candidates
        beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]
        if not beams:
            break

    all_candidates = completed + [(s, c) for s, c, _ in beams]
    _, best_caption = max(all_candidates, key=lambda x: x[0])

    return [vocab.itos[idx] for idx in best_caption
            if vocab.itos[idx] not in {"<SOS>", "<EOS>"}]


# ── 10. Trainer ───────────────────────────────────────────────────────────────
class CaptionTrainer:
    """
    Encapsulates training, validation, checkpointing and prediction.

    Args:
        model:           ``CNNtoRNN`` instance.
        train_dataset:   Training split of ``ImageCaptionDataset``.
        val_dataset:     Validation split.
        device:          Torch device string/object.
        vocab:           Shared ``Vocabulary``.
        transform:       Image transform (used during inference).
        checkpoint_path: Where to persist the latest checkpoint.
    """

    def __init__(self, model, train_dataset, val_dataset, device, vocab, transform,
                 checkpoint_path: str = "checkpoint.pth.tar"):
        self.model           = model.to(device)
        self.train_dataset   = train_dataset
        self.val_dataset     = val_dataset
        self.device          = device
        self.vocab           = vocab
        self.transform       = transform
        self.checkpoint_path = checkpoint_path

        self.pad_idx       = vocab.stoi["<PAD>"]
        self.criterion     = nn.CrossEntropyLoss(ignore_index=self.pad_idx)
        self.best_val_loss = float("inf")
        self.start_epoch   = 0

    # ── Checkpoint helpers ────────────────────────────────────────────────────
    def save_checkpoint(self, epoch: int, optimizer, is_best: bool = False):
        state = {
            "epoch":         epoch + 1,
            "state_dict":    self.model.state_dict(),
            "optimizer":     optimizer.state_dict(),
            "best_val_loss": self.best_val_loss,
        }
        torch.save(state, self.checkpoint_path)
        if is_best:
            torch.save(state, "best_model.pth.tar")
            print(f"  ✓ New best model saved (val loss {self.best_val_loss:.4f})")

    def load_checkpoint(self, optimizer) -> bool:
        if not os.path.exists(self.checkpoint_path):
            return False
        print("=> Loading checkpoint …")
        ckpt = torch.load(self.checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt["state_dict"])
        optimizer.load_state_dict(ckpt["optimizer"])
        self.start_epoch   = ckpt["epoch"]
        self.best_val_loss = ckpt["best_val_loss"]
        return True

    # ── Core loops ────────────────────────────────────────────────────────────
    def _loss_on_batch(self, imgs, caps):
        """Shared loss computation for training and evaluation."""
        outputs = self.model(imgs, caps[:, :-1])              # [B, T, V]
        # align: drop position-0 (image feature step), compare to cap[1:]
        logits  = outputs[:, 1:, :].reshape(-1, outputs.shape[2])
        targets = caps[:, 1:].reshape(-1)
        return self.criterion(logits, targets)

    def train(self, num_epochs: int = 20, batch_size: int = 32,
              lr: float = 3e-4, patience: int = 5, resume: bool = False):
        """
        Train the model with optional early stopping.

        Args:
            num_epochs: Total number of training epochs.
            batch_size: Samples per mini-batch.
            lr:         Adam learning rate.
            patience:   Early-stopping patience (epochs without improvement).
            resume:     If ``True``, try to resume from ``checkpoint_path``.
        """
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        if resume:
            self.load_checkpoint(optimizer)

        collate    = MyCollate(self.pad_idx)
        train_loader = DataLoader(self.train_dataset, batch_size=batch_size,
                                  shuffle=True,  collate_fn=collate)
        val_loader   = DataLoader(self.val_dataset,   batch_size=batch_size,
                                  shuffle=False, collate_fn=collate)

        epochs_no_improve = 0
        print(f"Training from epoch {self.start_epoch + 1} to {num_epochs} …\n")

        for epoch in range(self.start_epoch, num_epochs):
            # ── Train ────────────────────────────────────────────────────────
            self.model.train()
            total_train_loss = 0.0
            for imgs, caps in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [train]",
                                   leave=False):
                imgs, caps = imgs.to(self.device), caps.to(self.device)
                loss = self._loss_on_batch(imgs, caps)
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                total_train_loss += loss.item()

            avg_train_loss = total_train_loss / len(train_loader)

            # ── Validate ─────────────────────────────────────────────────────
            avg_val_loss = self._evaluate_loader(val_loader)

            print(f"Epoch [{epoch+1:>3}/{num_epochs}] | "
                  f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

            # ── Checkpoint & early stopping ───────────────────────────────────
            is_best = avg_val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss  = avg_val_loss
                epochs_no_improve   = 0
            else:
                epochs_no_improve  += 1

            self.save_checkpoint(epoch, optimizer, is_best)

            if epochs_no_improve >= patience:
                print(f"\nEarly stopping triggered after {patience} epochs with no improvement.")
                break

    def _evaluate_loader(self, loader) -> float:
        """Return average cross-entropy loss over *loader*."""
        self.model.eval()
        total = 0.0
        with torch.no_grad():
            for imgs, caps in loader:
                imgs, caps = imgs.to(self.device), caps.to(self.device)
                total += self._loss_on_batch(imgs, caps).item()
        return total / len(loader)

    def predict(self, image, beam_size: int = 5, max_length: int = 50) -> list[str]:
        """
        Generate a caption for a PIL image using beam search.

        Returns:
            List of word strings (without special tokens).
        """
        return generate_caption_beam_search(
            image, self.model, self.transform, self.vocab,
            self.device, beam_size=beam_size, max_length=max_length,
        )


# ── 11. Evaluation (test set) ─────────────────────────────────────────────────
def evaluate_test_set(model, test_dataset, vocab, transform, device):
    """
    Compute cross-entropy loss and corpus BLEU-4 on *test_dataset*.

    BLEU is computed using full beam-search inference (not teacher forcing).

    Args:
        model:        Trained ``CNNtoRNN``.
        test_dataset: ``ImageCaptionDataset`` test split.
        vocab:        Shared ``Vocabulary``.
        transform:    Image transform used during inference.
        device:       Torch device.

    Returns:
        Tuple ``(avg_loss, bleu_score)``.
    """
    model.eval()
    criterion   = nn.CrossEntropyLoss(ignore_index=vocab.stoi["<PAD>"])
    test_loader = DataLoader(
        test_dataset, batch_size=1, shuffle=False,
        collate_fn=MyCollate(vocab.stoi["<PAD>"]),
    )

    total_loss = 0.0
    candidates: list[str]       = []   # torchmetrics expects joined strings
    references: list[list[str]] = []   # list of reference-sets per sample

    with torch.no_grad():
        for imgs, caps in tqdm(test_loader, desc="Evaluating test set"):
            imgs, caps = imgs.to(device), caps.to(device)

            # Teacher-forcing loss
            outputs = model(imgs, caps[:, :-1])
            logits  = outputs[:, 1:, :].reshape(-1, outputs.shape[2])
            targets = caps[:, 1:].reshape(-1)
            total_loss += criterion(logits, targets).item()

            # Beam-search caption (pass tensor directly — already transformed)
            predicted_words = generate_caption_beam_search(
                imgs[0], model, transform, vocab, device
            )
            target_words = [
                vocab.itos[idx]
                for idx in caps[0].tolist()
                if vocab.itos[idx] not in {"<SOS>", "<EOS>", "<PAD>"}
            ]

            # torchmetrics bleu_score expects List[str] candidates,
            # List[List[str]] references (one list of refs per sample)
            candidates.append(" ".join(predicted_words))
            references.append([" ".join(target_words)])

    avg_loss = total_loss / len(test_loader)
    score    = bleu_score(candidates, references, n_gram=4)

    print(f"\nTest Loss: {avg_loss:.4f} | BLEU-4: {score:.4f}")
    return avg_loss, score


In [7]:
# ── Data ──────────────────────────────────────────────────────────────────
captions_df = pd.read_csv("flickr8k/captions.txt")

train_val_df, test_df  = train_test_split(captions_df, test_size=0.1,  random_state=42)
train_df,     val_df   = train_test_split(train_val_df, test_size=0.1, random_state=42)

vocab = Vocabulary(freq_threshold=5)
vocab.build_vocabulary(captions_df["caption"].tolist())

# Save vocab to pickle file
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)


vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 3005


In [ ]:
train_dataset = ImageCaptionDataset("flickr8k/Images", train_df, transform, vocab)
val_dataset   = ImageCaptionDataset("flickr8k/Images", val_df,   transform, vocab)
test_dataset  = ImageCaptionDataset("flickr8k/Images", test_df,  transform, vocab)

# show_examples(train_dataset)

# ── Model ─────────────────────────────────────────────────────────────────
EMBED_SIZE  = 256
HIDDEN_SIZE = 512   # bumped from 256 — gives the decoder more capacity
NUM_LAYERS  = 1

model = CNNtoRNN(EMBED_SIZE, HIDDEN_SIZE, vocab_size, NUM_LAYERS)

# ── Train ─────────────────────────────────────────────────────────────────
trainer = CaptionTrainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    device=device,
    vocab=vocab,
    transform=transform,
)

trainer.train(num_epochs=20, batch_size=32, lr=3e-4, patience=5)


Vocabulary size: 3005
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 213MB/s]


Training from epoch 1 to 20 …



Epoch [  1/20] | Train Loss: 3.9006 | Val Loss: 3.3564
  ✓ New best model saved (val loss 3.3564)


Epoch [  2/20] | Train Loss: 3.2456 | Val Loss: 3.0611
  ✓ New best model saved (val loss 3.0611)


...

Epoch [ 16/20] | Train Loss: 2.2437 | Val Loss: 2.5383
  ✓ New best model saved (val loss 2.5383)


Epoch [ 17/20] | Train Loss: 2.2194 | Val Loss: 2.5323
  ✓ New best model saved (val loss 2.5323)


Epoch [ 18/20] | Train Loss: 2.1950 | Val Loss: 2.5325


Epoch [ 19/20] | Train Loss: 2.1693 | Val Loss: 2.5263
  ✓ New best model saved (val loss 2.5263)


Epoch [ 20/20] | Train Loss: 2.1453 | Val Loss: 2.5300


In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────
image = Image.open("child.jpg")
caption = trainer.predict(image)
print("Caption:", " ".join(caption))

Caption: a little girl in a pink shirt is swinging on a swing .


In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────
image = Image.open("dog.jpg")
caption = trainer.predict(image)
print("Caption:", " ".join(caption))

Caption: a brown dog is running on the beach .


In [ ]:
# ── Test ──────────────────────────────────────────────────────────────────
evaluate_test_set(model, test_dataset, vocab, transform, device)

In [ ]:
import pickle
import torch
from torch import nn
import numpy as np
from tqdm import tqdm


device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/adityajn105/flickr8k")

In [ ]:
import torchvision.transforms as transforms


# Define the transform pipeline
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],    # Standard ImageNet mean
        std=[0.229, 0.224, 0.225]      # Standard ImageNet std
    )
])


In [ ]:
import torch
from collections import Counter


class Vocabulary:
    """
    handles word-to-index mapping and should be saved as
    a .pkl file so you can swap it when changing datasets.

    Args:
        freq_threshold (int): minimum word frequency to be included
        in the vocabulary
    """
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {v: k for k, v in self.itos.items()}
        self.freq_threshold = freq_threshold

    def build_vocabulary(self, sentence_list):
        """Build the vocabulary from a list of sentences."""
        frequencies = Counter()
        idx = 4 # Start Index for new words to the stoi and itos dictionaries
        for sentence in sentence_list:
            for word in sentence.lower().split():
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        """Numericalize the text."""
        tokenized_text = text.lower().split()
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokenized_text]


In [ ]:
import os
import torch
from PIL import Image
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence


class ImageCaptionDataset(Dataset):
    """Dataset with flexible loading for Flickr8k/30k/MS COCO.

    Args:
        root_dir (str): Directory where images are stored.
        captions_df (pd.DataFrame): DataFrame containing image-caption pairs.
        transform (callable): A function/transform that takes in an PIL image
            and returns a transformed version.
        vocab (Vocabulary): Vocabulary object for word-to-index mapping.
    """
    def __init__(self, root_dir, captions_df, transform=None, vocab=None):
        self.root_dir = root_dir
        self.df = captions_df
        self.transform = transform
        self.vocab = vocab
        if vocab is None:
            self.vocab = Vocabulary(freq_threshold=5)
            self.vocab.build_vocabulary(self.df["caption"].tolist())

    def __len__(self): return len(self.df)

    def __getitem__(self, index):
        """
        Args:
            index (int): Index of the item to retrieve.

        Returns:
            tuple: (image, numericalized_caption)
        """
        caption = self.df.iloc[index]["caption"]
        img_id = self.df.iloc[index]["image"]
        img = Image.open(os.path.join(self.root_dir, img_id)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        # Numericalize caption and add start/end tokens
        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption += [self.vocab.stoi["<EOS>"]]

        return img, torch.tensor(numericalized_caption)



class MyCollate:
    """Pads batches dynamically.

    Args:
        pad_idx (int): Index of the padding token in the vocabulary.
    """
    def __init__(self, pad_idx): self.pad_idx = pad_idx

    def __call__(self, batch):
        """
        Collate function for DataLoader.
        That is it is called on each batch of data from the
        DataLoader to pad the captions so they are all the same length.

        Args:
            batch (list): List of (image, caption) tuples.

        Returns:
            tuple: (images, targets)
        """
        imgs = torch.stack([item[0] for item in batch])

        """ pad_sequence -> This function is used to pad the captions to the same length.
        input -> A list of tensors to be padded.
        batch_first -> If True, the returned tensor will have the batch dimension first.
        padding_value -> The value to use for padding, that is the index of
            the padding token in the vocabulary, in this case <PAD>.
            ie vocab.stoi["<PAD>"]
        """
        targets = pad_sequence([item[1] for item in batch], batch_first=True, padding_value=self.pad_idx)
        return imgs, targets


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def show_examples(dataset, num = 9):
    """Show a few examples from the dataset."""
    img_capt = []
    plt.figure(figsize=(18, 15))

    for i in range(num):
        img, caption = dataset[i]
        img_display = img.permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_display = std * img_display + mean
        img_display = np.clip(img_display, 0, 1) # Ensure values are 0-1

        readable_caption = [dataset.vocab.itos[token.item()] for token in caption]
        clean_caption = " ".join([w for w in readable_caption if w not in ["<PAD>", "<SOS>", "<EOS>"]])

        img_capt.append((img_display, clean_caption))

    # plot the example in img_capt
    for i in range(num):
        plt.subplot(3, 3, i+1)
        plt.imshow(img_capt[i][0])
        plt.title(img_capt[i][1])
        plt.axis("off")
    plt.show()




In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


# Read the captions file into a DataFrame
captions_df = pd.read_csv("flickr8k/captions.txt")

# Create Train, Test, Validate dataset
train_test_df, val_df = train_test_split(captions_df, test_size=0.2, random_state=42)
train_df, test_df = train_test_split(train_test_df, test_size=0.2)

# Initialize and build vocabulary
vocab = Vocabulary(freq_threshold=5)
vocab.build_vocabulary(captions_df["caption"].tolist())

vocab_size = len(vocab.stoi)
print(f"Vocab size: {vocab_size}")

# General Dataset
gen_dataset = ImageCaptionDataset(
    root_dir="flickr8k/Images",
    captions_df = captions_df,
    transform=transform,
    vocab=vocab
)

# Create Train, Test, Validate Datasets
train_dataset = ImageCaptionDataset(
    root_dir="flickr8k/Images",
    captions_df=train_df,
    transform=transform,
    vocab=vocab
)
test_dataset = ImageCaptionDataset(
    root_dir="flickr8k/Images",
    captions_df=test_df,
    transform=transform,
    vocab=vocab
)

val_dataset = ImageCaptionDataset(
    root_dir="flickr8k/Images",
    captions_df=val_df,
    transform=transform,
    vocab=vocab
)


Vocab size: 3005


In [ ]:
# show_examples(train_dataset)

In [ ]:
from torchvision import models


class EncoderCNN(nn.Module):
    """
    CNN Encoder Model

    We use a pre-trained ResNet50. We freeze the weights and
    replace the final FC layer with a linear layer that matches our hidden size.

    Args:
        embed_size (int): Dimensionality of the word embedding.
            ie vocab_size which is the length of the vocabulary.
        train_CNN (bool): Whether to train the CNN model.
    """
    def __init__(self, embed_size, train_CNN=False):
        super(EncoderCNN, self).__init__()
        self.train_CNN = train_CNN
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, embed_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, images):
        """
        Returns the features of the image. shape is [batch, embed_size]
        """
        features = self.resnet(images)
        return self.dropout(self.relu(features))


In [ ]:
class DecoderRNN(nn.Module):
    """
    This takes the image features and generates word sequences.

    Args:
        embed_size (int): Dimensionality of the word embedding.
            ie vocab_size which is the length of the vocabulary.
        hidden_size (int): Dimensionality of the hidden state.
        vocab_size (int): Size of the vocabulary.
        num_layers (int): Number of layers in the LSTM.
    """
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers):
        super(DecoderRNN, self).__init__()

        """ self.embed -> An embedding layer that maps each word in the vocabulary
        to a dense vector of size embed_size.
        """
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers)
        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):
        # features shape: [batch, embed_size]
        # captions shape: [batch, seq_len] (from MyCollate with batch_first=True)
        embeddings = self.dropout(self.embed(captions))

        # Prepend the image features to the caption embeddings
        # features.unsqueeze(1) changes [batch, embed_size] to [batch, 1, embed_size]
        # Concatenate along dim=1 (sequence dimension) to get [batch, 1 + seq_len, embed_size]
        embeddings = torch.cat((features.unsqueeze(1), embeddings), dim=1)

        hiddens, _ = self.lstm(embeddings)
        outputs = self.linear(hiddens)
        return outputs

In [ ]:
class CNNtoRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers=1):
        super(CNNtoRNN, self).__init__()
        self.encoderCNN = EncoderCNN(embed_size)
        self.decoderRNN = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers)

    def forward(self, images, captions):
        features = self.encoderCNN(images)
        outputs = self.decoderRNN(features, captions)
        return outputs


In [ ]:
import torch
import torch.nn as nn
import os
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

try:
    from torchmetrics.functional.text import bleu_score
except ModuleNotFoundError:
    !pip install torchmetrics --quiet
    from torchmetrics.functional.text import bleu_score

class CaptionTrainer:
    def __init__(self, model, train_dataset, val_dataset, device, vocab, transform,
                 checkpoint_path="checkpoint.pth.tar"):
        self.model = model.to(device)
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.device = device
        self.vocab = vocab
        self.transform = transform
        self.checkpoint_path = checkpoint_path

        self.pad_idx = vocab.stoi["<PAD>"]
        self.criterion = nn.CrossEntropyLoss(ignore_index=self.pad_idx)
        self.best_val_loss = float("inf")
        self.start_epoch = 0

    def save_checkpoint(self, epoch, optimizer, is_best=False):
        state = {
            "epoch": epoch + 1,
            "state_dict": self.model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_val_loss": self.best_val_loss,
        }
        torch.save(state, self.checkpoint_path)
        if is_best:
            torch.save(state, "best_model.pth.tar")

    def load_checkpoint(self, optimizer):
        if os.path.exists(self.checkpoint_path):
            print("=> Loading checkpoint...")
            checkpoint = torch.load(self.checkpoint_path)
            self.model.load_state_dict(checkpoint["state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            self.start_epoch = checkpoint["epoch"]
            self.best_val_loss = checkpoint["best_val_loss"]
            return True
        return False

    def train(self, num_epochs=20, batch_size=32, lr=3e-4, patience=5):
        print("Training...")

        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        # self.load_checkpoint(optimizer)

        train_loader = DataLoader(self.train_dataset, batch_size=batch_size, shuffle=True,
                                  collate_fn=MyCollate(self.pad_idx))
        val_loader = DataLoader(self.val_dataset, batch_size=batch_size, shuffle=False,
                                collate_fn=MyCollate(self.pad_idx))

        # epochs_no_improve = 0

        print(f"Start EPOCH: {self.start_epoch}  ->  Number of EPOCHS: {num_epochs} ")
        for epoch in range(self.start_epoch, num_epochs):
            # Training
            self.model.train()
            total_train_loss = 0
            for imgs, caps in train_loader:
                imgs, caps = imgs.to(self.device), caps.to(self.device)
                outputs = self.model(imgs, caps[:, :-1])

                # Fix: Slice outputs to match the target caption length
                loss = self.criterion(outputs[:, 1:, :].reshape(-1, outputs.shape[2]), caps[:, 1:].reshape(-1))
                # print(f"output shape: {outputs.shape}")

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

                if (epoch+1) % 5 == 0:
                    print(f"Epoch [{epoch+1}/{num_epochs}] | Batch Loss: {loss.item():.4f}")

            print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {total_train_loss/len(train_loader):.4f}")

            # Validation
            # avg_val_loss = self.evaluate(val_loader)
            # print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {total_train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f}")

            # Checkpointing & Early Stopping
            # is_best = avg_val_loss < self.best_val_loss
            # if is_best:
            #     self.best_val_loss = avg_val_loss
            #     epochs_no_improve = 0
            # else:
            #     epochs_no_improve += 1

            # self.save_checkpoint(epoch, optimizer, is_best)

            # if epochs_no_improve >= patience:
            #     print("Early stopping triggered.")
            #     break

    def evaluate(self, loader):
        self.model.eval()
        total_loss = 0
        with torch.no_grad():
            for imgs, caps in loader:
                imgs, caps = imgs.to(self.device), caps.to(self.device)
                outputs = self.model(imgs, caps[:, :-1])
                # Fix: Slice outputs to match the target caption length
                loss = self.criterion(outputs[:, 1:, :].reshape(-1, outputs.shape[2]), caps[:, 1:].reshape(-1))
                total_loss += loss.item()
        return total_loss / len(loader)

    def predict_and_visualize(self, image, beam_size=5, max_length=50):
        # Implementation of your generate_caption logic inside the class
        self.model.eval()
        # 1. Image Preprocessing & Feature Extraction
        img = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            # Get image features from Encoder
            features = self.model.encoderCNN(img).unsqueeze(1) # (1, 1, embed_size)

            # Start the beam with <SOS>
            # Each beam item: (sequence_indices, log_probability, hidden_states)
            start_token = vocab.stoi["<SOS>"]
            initial_cap = [start_token]

            # Initialize hidden/cell states with features (if your model does this)
            # Or pass features as the first input
            _, states = self.model.decoderRNN.lstm(features)

            # beams list stores tuples: (score, current_caption, last_states)
            beams = [(0.0, initial_cap, states)]
            completed_captions = []

            for _ in range(max_length):
                new_beams = []

                for score, caption, last_states in beams:
                    # Get the last word predicted in this beam
                    last_word = torch.tensor([caption[-1]]).to(device)
                    embeds = self.model.decoderRNN.embed(last_word).unsqueeze(0)

                    # Pass through LSTM
                    output, current_states = self.model.decoderRNN.lstm(embeds, last_states)
                    logits = self.model.decoderRNN.linear(output.squeeze(0))

                    # Convert to log probabilities
                    log_probs = F.log_softmax(logits, dim=1)

                    # Get top K candidates for this specific beam
                    top_k_log_probs, top_k_idx = log_probs.topk(beam_size)

                    for i in range(beam_size):
                        next_word = top_k_idx[0][i].item()
                        new_score = score + top_k_log_probs[0][i].item()
                        new_caption = caption + [next_word]

                        if next_word == vocab.stoi["<EOS>"]:
                            completed_captions.append((new_score, new_caption))
                        else:
                            new_beams.append((new_score, new_caption, current_states))

                # Keep only the top 'beam_size' candidates overall
                beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]

                # If all active beams are empty (all reached <EOS>), stop
                if not beams:
                    break

            # Final selection: combine completed and active beams, then pick the best
            all_candidates = completed_captions + [(s, c) for s, c, st in beams]
            best_score, best_caption = max(all_candidates, key=lambda x: x[0])

            # Convert indices to words, removing <SOS> and <EOS>
            return [vocab.itos[idx] for idx in best_caption if vocab.itos[idx] not in ["<SOS>", "<EOS>"]]

In [ ]:
from torch.optim import optimizer
embed_size = 256
hidden_size = 256
num_layers = 1


# Initialize the model
model = CNNtoRNN(embed_size, hidden_size, vocab_size)

# Train, validate, test, and predict
caption_trainer = CaptionTrainer(
    model=model,
    train_dataset=gen_dataset,
    val_dataset=val_dataset,
    device=device,
    vocab=vocab,
    transform=transform
)

caption_trainer.train()

In [ ]:
# Prdict
image = Image.open("child.jpg")

prediction = caption_trainer.predict_and_visualize(image)

print(f"Prediction: {prediction}")

In [ ]:
import torch
import torch.nn.functional as F

def generate_caption_beam_search(
        image, model, transform, vocab,
        device, beam_size=5, max_length=50
    ):
    """
    Generate a caption for an image using the trained model.

    Args:
        image (PIL.Image): The input image.
        model (nn.Module): The trained model.
        transform (callable): The transformation to apply to the image.
        vocab (Vocabulary): The vocabulary used for the dataset.
        device (torch.device): The device to use for inference.
        beam_size (int): The size of the beam for beam search.
            ie how many candidates to keep at each step.
        max_length (int): The maximum length of the generated caption.

    Returns:
        str: The generated caption.
    """

    model.eval()

    # 1. Image Preprocessing & Feature Extraction
    img = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        # Get image features from Encoder
        features = model.encoderCNN(img).unsqueeze(1) # (1, 1, embed_size)

        # Start the beam with <SOS>
        # Each beam item: (sequence_indices, log_probability, hidden_states)
        start_token = vocab.stoi["<SOS>"]
        initial_cap = [start_token]

        # Initialize hidden/cell states with features (if your model does this)
        # Or pass features as the first input
        _, states = model.decoderRNN.lstm(features)

        # beams list stores tuples: (score, current_caption, last_states)
        beams = [(0.0, initial_cap, states)]
        completed_captions = []

        for _ in range(max_length):
            new_beams = []

            for score, caption, last_states in beams:
                # Get the last word predicted in this beam
                last_word = torch.tensor([caption[-1]]).to(device)
                embeds = model.decoderRNN.embed(last_word).unsqueeze(0)

                # Pass through LSTM
                output, current_states = model.decoderRNN.lstm(embeds, last_states)
                logits = model.decoderRNN.linear(output.squeeze(0))

                # Convert to log probabilities
                log_probs = F.log_softmax(logits, dim=1)

                # Get top K candidates for this specific beam
                top_k_log_probs, top_k_idx = log_probs.topk(beam_size)

                for i in range(beam_size):
                    next_word = top_k_idx[0][i].item()
                    new_score = score + top_k_log_probs[0][i].item()
                    new_caption = caption + [next_word]

                    if next_word == vocab.stoi["<EOS>"]:
                        completed_captions.append((new_score, new_caption))
                    else:
                        new_beams.append((new_score, new_caption, current_states))

            # Keep only the top 'beam_size' candidates overall
            beams = sorted(new_beams, key=lambda x: x[0], reverse=True)[:beam_size]

            # If all active beams are empty (all reached <EOS>), stop
            if not beams:
                break

        # Final selection: combine completed and active beams, then pick the best
        all_candidates = completed_captions + [(s, c) for s, c, st in beams]
        best_score, best_caption = max(all_candidates, key=lambda x: x[0])

        # Convert indices to words, removing <SOS> and <EOS>
        return [vocab.itos[idx] for idx in best_caption if vocab.itos[idx] not in ["<SOS>", "<EOS>"]]


# To use it:
# caption_list = generate_caption_beam_search(image, model, transform, dataset.vocab, device)
# print(" ".join(caption_list))

In [ ]:
caption_list = generate_caption_beam_search(image, model, transform, gen_dataset.vocab, device)
print(" ".join(caption_list))

In [ ]:
from torchmetrics.functional.text import bleu_score


def test(model, test_dataset, device, vocab, loss_fn, transform):
    """
    Test the model on the test dataset.

    Args:
        model (nn.Module): The trained model.
        test_dataset (ImageCaptionDataset): The test dataset.
        device (torch.device): The device to use for testing.
        vocab (Vocabulary): The vocabulary used for the dataset.
        lost_fn (callable): The loss function to use.
    """
    model.eval()
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=1,
        shuffle=False,
        collate_fn=MyCollate(pad_idx=vocab.stoi["<PAD>"])
    )

    total_loss = 0
    references = []
    candidates = []

    with torch.no_grad():
        for imgs, captions in test_loader:
            imgs, captions = imgs.to(device), captions.to(device)

            # 1. Calculate Loss (Teacher Forcing)
            outputs = model(imgs, captions[:, :-1])
            # Reshape: (batch * seq_len, vocab_size) and (batch * seq_len)
            loss = loss_fn(outputs.reshape(-1, outputs.shape[2]), captions[:, 1:].reshape(-1))
            total_loss += loss.item()

            # 2. Calculate BLEU (Inference)
            # We generate a caption word-by-word to see how it actually performs
            predicted_tokens = generate_caption_beam_search(imgs[0], model, transform, vocab, device)

            # Target caption: remove special tokens and convert to list of strings
            target_tokens = [vocab.itos[idx] for idx in captions[0].tolist()
                             if vocab.itos[idx] not in ["<SOS>", "<EOS>", "<PAD>"]]

            candidates.append(predicted_tokens)
            references.append([target_tokens]) # BLEU expects a list of possible references

    avg_loss = total_loss / len(test_loader)
    score = bleu_score(candidates, references)

    print(f"Test Loss: {avg_loss:.4f} | BLEU Score: {score:.4f}")
    return avg_loss, score
